In [ ]:

# =============================================================================
# Google Ads — Gold star schema (Development)
# Silver : Files/Development/Silver/GoogleAds/
# Gold   : Files/Development/Gold/GoogleAds/  + schema Gold
# Also creates unified Meta + Google view: Gold.vw_unified_ad_performance
# =============================================================================

from pyspark.sql import functions as F, Window

WORKSPACE_ID = "718e8176-5d40-4a9c-88ff-50ac97ac49ba"
LAKEHOUSE_ID = "981fbe98-2f01-41d8-bf2f-a85e5cd9e2a2"
BASE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
SILVER_PATH = f"{BASE_PATH}/Files/Development/Silver/GoogleAds"
GOLD_PATH = f"{BASE_PATH}/Files/Development/Gold/GoogleAds"
GOLD_SCHEMA = "Gold"

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
FULL_REFRESH = False
INCREMENTAL_LOOKBACK_DAYS = 2
INCREMENTAL_MAX_DAYS = 14
print("[RUNTIME]", SILVER_PATH, "->", GOLD_PATH, "FULL_REFRESH=", FULL_REFRESH)

# Incremental helpers (controls set above)
from delta.tables import DeltaTable

def _table_exists(name: str) -> bool:
    try:
        spark.table(name).limit(1).collect()
        return True
    except Exception:
        return False

def _path_is_delta(path: str) -> bool:
    try:
        return DeltaTable.isDeltaTable(spark, path)
    except Exception:
        return False

def _max_date(table_or_path: str, date_col: str, is_path: bool = False):
    try:
        df = spark.read.format("delta").load(table_or_path) if is_path else spark.table(table_or_path)
        return df.agg(F.max(F.col(date_col)).alias("m")).collect()[0]["m"]
    except Exception:
        return None

def filter_by_watermark(df, date_col: str, target: str, is_path: bool = False):
    if FULL_REFRESH:
        print(f"[FULL_REFRESH] no watermark filter: {target}")
        return df
    exists = _path_is_delta(target) if is_path else _table_exists(target)
    if not exists:
        print(f"[INCR] target missing → first load: {target}")
        return df
    wm = _max_date(target, date_col, is_path=is_path)
    if wm is None:
        print(f"[INCR] empty watermark → full batch: {target}")
        return df
    out = df.filter(F.col(date_col).isNotNull() & (F.col(date_col) >= F.date_sub(F.lit(wm), int(INCREMENTAL_LOOKBACK_DAYS))))
    st = out.agg(F.min(date_col).alias("mn"), F.max(date_col).alias("mx"), F.count(F.lit(1)).alias("n")).collect()[0]
    print(f"[INCR] {target} wm={wm} lookback={INCREMENTAL_LOOKBACK_DAYS}d rows={st['n']} range={st['mn']}..{st['mx']}")
    if st["n"] and st["mn"] is not None and st["mx"] is not None:
        span = (st["mx"] - st["mn"]).days
        if span > int(INCREMENTAL_MAX_DAYS):
            raise ValueError(
                f"Incremental batch for {target} spans {span} days (> {INCREMENTAL_MAX_DAYS}). "
                "Refusing large backfill. Use daily/2-day bronze, or set FULL_REFRESH=True intentionally."
            )
    return out

def merge_or_overwrite_table(df, target: str, keys, partition_cols=None, stamp_col="gold_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _table_exists(target):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.saveAsTable(target)
        mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "MERGE"
    print(f"[OK] {target} ({mode}) total={spark.table(target).count():,}")

def merge_or_overwrite_path(df, path: str, keys, partition_cols=None, stamp_col="_processed_at"):
    if stamp_col and stamp_col not in df.columns:
        df = df.withColumn(stamp_col, F.current_timestamp())
    df = df.dropDuplicates(list(keys))
    if FULL_REFRESH or not _path_is_delta(path):
        w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").option("mergeSchema", "true")
        if partition_cols:
            w = w.partitionBy(*partition_cols)
        w.save(path)
        mode = "OVERWRITE"
    else:
        cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        (DeltaTable.forPath(spark, path).alias("t").merge(df.alias("s"), cond)
         .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        mode = "MERGE"
    print(f"[OK] {path} ({mode}) total={spark.read.format('delta').load(path).count():,}")


def write_both(df, table, rel_path, keys, partition_cols=None, date_col=None):
    path = f"{GOLD_PATH}/{rel_path}"
    target = f"{GOLD_SCHEMA}.{table}"
    batch = df
    if date_col:
        batch = filter_by_watermark(batch, date_col, target)
        if len(batch.take(1)) == 0:
            print(f"[SKIP] {target}: no new incremental rows")
            return
    merge_or_overwrite_table(batch, target, keys, partition_cols=partition_cols, stamp_col="gold_processed_at")
    merge_or_overwrite_path(batch, path, keys, partition_cols=partition_cols, stamp_col="gold_processed_at")

campaigns = spark.read.format("delta").load(f"{SILVER_PATH}/silver_google_campaigns")
adgroups  = spark.read.format("delta").load(f"{SILVER_PATH}/silver_google_adgroups")
ads       = spark.read.format("delta").load(f"{SILVER_PATH}/silver_google_ads")
ad_perf   = spark.read.format("delta").load(f"{SILVER_PATH}/silver_google_ad_performance")
print("[INFO]", campaigns.count(), adgroups.count(), ads.count(), ad_perf.count())

# Dimensions (google-prefixed tables to avoid clobbering Meta dims)
dim_account = (
    campaigns.select("account_id","account_name","platform","tenant_id").dropDuplicates(["account_id"])
    .withColumn("account_sk", F.row_number().over(Window.orderBy("account_id")))
    .withColumn("gold_processed_at", F.current_timestamp())
)
write_both(dim_account, "dim_google_account", "dim_account", keys=["account_id"])

dim_campaign = (
    campaigns.withColumn("campaign_sk", F.row_number().over(Window.orderBy("campaign_id")))
    .join(dim_account.select("account_id","account_sk"), "account_id", "left")
    .select(
        "campaign_sk","campaign_id","account_sk","campaign_name","status",
        F.col("channel_type").alias("channel_type"),
        F.col("daily_budget_inr"),
        F.current_timestamp().alias("gold_processed_at"),
    )
)
write_both(dim_campaign, "dim_google_campaign", "dim_campaign", keys=["campaign_id"])

dim_adgroup = (
    adgroups.withColumn("adgroup_sk", F.row_number().over(Window.orderBy("adgroup_id")))
    .join(dim_campaign.select("campaign_id","campaign_sk"), "campaign_id", "left")
    .select(
        "adgroup_sk","adgroup_id","campaign_sk","adgroup_name","status",
        F.col("cpc_bid_inr"),
        F.current_timestamp().alias("gold_processed_at"),
    )
)
write_both(dim_adgroup, "dim_google_adgroup", "dim_adgroup", keys=["adgroup_id"])

dim_ad = (
    ads.withColumn("ad_sk", F.row_number().over(Window.orderBy("ad_id")))
    .join(dim_adgroup.select("adgroup_id","adgroup_sk"), "adgroup_id", "left")
    .join(dim_campaign.select("campaign_id","campaign_sk"), "campaign_id", "left")
    .select(
        "ad_sk","ad_id","adgroup_sk","campaign_sk","ad_type","status","headline","description","final_urls",
        F.current_timestamp().alias("gold_processed_at"),
    )
)
write_both(dim_ad, "dim_google_ad", "dim_ad", keys=["ad_id"])

dim_date = (
    ad_perf.select(F.col("date").alias("full_date")).where(F.col("full_date").isNotNull()).dropDuplicates()
    .select(
        F.date_format("full_date","yyyyMMdd").cast("int").alias("date_key"),
        "full_date",
        F.year("full_date").alias("year"), F.month("full_date").alias("month"), F.dayofmonth("full_date").alias("day"),
        F.date_format("full_date","MMMM").alias("month_name"),
        F.dayofweek("full_date").alias("day_of_week"),
        F.date_format("full_date","EEEE").alias("day_name"),
        F.current_timestamp().alias("gold_processed_at"),
    )
)
write_both(dim_date, "dim_google_date", "dim_date", keys=["date_key"])

fact = (
    ad_perf.join(dim_ad.select("ad_id","ad_sk","adgroup_sk","campaign_sk"), "ad_id", "left")
    .join(dim_account.select("account_id","account_sk"), "account_id", "left")
    .select(
        F.date_format(F.col("date"),"yyyyMMdd").cast("int").alias("date_key"),
        "account_sk","campaign_sk","adgroup_sk","ad_sk",
        F.col("impressions").cast("double"), F.col("clicks").cast("double"),
        F.col("ctr").cast("double"), F.col("spend_inr").cast("double"),
        F.col("average_cpc").cast("double").alias("cpc"),
        F.col("conversions").cast("double"), F.col("conversions_value").cast("double"),
        F.col("cost_per_conversion").cast("double"), F.col("roas").cast("double"),
        F.lit(None).cast("double").alias("engagements"), F.lit(None).cast("double").alias("video_views"),
    )
    .where(F.col("ad_sk").isNotNull() & F.col("date_key").isNotNull())
    .withColumn("fact_sk", F.row_number().over(Window.orderBy("date_key","ad_sk")))
    .withColumn("gold_processed_at", F.current_timestamp())
)
write_both(fact, "fact_google_ad_performance_daily", "fact_ad_performance_daily", keys=["ad_sk", "date_key"], partition_cols=["date_key"])

# Reporting table (denormalized) — includes campaign/adgroup/ad attributes
rpt = (
    spark.table(f"{GOLD_SCHEMA}.fact_google_ad_performance_daily").alias("f")
    .join(spark.table(f"{GOLD_SCHEMA}.dim_google_date").alias("d"), "date_key")
    .join(spark.table(f"{GOLD_SCHEMA}.dim_google_account").alias("a"), "account_sk")
    .join(spark.table(f"{GOLD_SCHEMA}.dim_google_campaign").alias("c"), "campaign_sk")
    .join(spark.table(f"{GOLD_SCHEMA}.dim_google_adgroup").alias("g"), "adgroup_sk")
    .join(spark.table(f"{GOLD_SCHEMA}.dim_google_ad").alias("ad"), "ad_sk")
    .select(
        F.lit("google_ads").alias("platform"),
        F.col("d.full_date"), F.col("d.year"), F.col("d.month"), F.col("d.month_name"), F.col("d.day_name"),
        F.col("a.account_id"), F.col("a.account_name"),
        F.col("c.campaign_id"), F.col("c.campaign_name"), F.col("c.status").alias("campaign_status"),
        F.col("c.channel_type").alias("campaign_channel_or_objective"),
        F.col("c.daily_budget_inr").alias("campaign_daily_budget_inr"),
        F.col("g.adgroup_id").alias("adset_or_adgroup_id"),
        F.col("g.adgroup_name").alias("adset_or_adgroup_name"),
        F.col("g.status").alias("adset_or_adgroup_status"),
        F.col("ad.ad_id"), F.coalesce(F.col("ad.headline"), F.col("ad.ad_type")).alias("ad_name"),
        F.col("ad.ad_type"), F.col("ad.status").alias("ad_status"), F.col("ad.final_urls"),
        F.col("f.impressions"), F.col("f.clicks"), F.col("f.ctr"), F.col("f.spend_inr"), F.col("f.cpc"),
        F.col("f.conversions"), F.col("f.conversions_value"), F.col("f.cost_per_conversion"), F.col("f.roas"),
        F.col("f.engagements"), F.col("f.video_views"),
        F.current_timestamp().alias("gold_processed_at"),
    )
)
write_both(rpt, "rpt_google_ad_performance_daily", "rpt_google_ad_performance_daily", keys=["account_id","campaign_id","adgroup_id","ad_id","full_date"], partition_cols=["full_date"], date_col="full_date")

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_google_ad_performance AS
SELECT * FROM {GOLD_SCHEMA}.rpt_google_ad_performance_daily
""")
print("[OK] Gold.vw_google_ad_performance")

# ---- Unified Meta + Google reporting view ----
# Align Meta rpt columns into the same shape
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.vw_unified_ad_performance AS
SELECT
  CAST('meta' AS STRING) AS platform,
  full_date, year, month, month_name, day_name,
  account_id, account_name,
  campaign_id, campaign_name, campaign_status,
  campaign_objective AS campaign_channel_or_objective,
  campaign_daily_budget_inr,
  adset_id AS adset_or_adgroup_id,
  adset_name AS adset_or_adgroup_name,
  adset_status AS adset_or_adgroup_status,
  ad_id, ad_name,
  CAST(NULL AS STRING) AS ad_type,
  ad_status,
  CAST(NULL AS STRING) AS final_urls,
  impressions, clicks, ctr, spend_inr, cpc,
  CAST(NULL AS DOUBLE) AS conversions,
  CAST(NULL AS DOUBLE) AS conversions_value,
  CAST(NULL AS DOUBLE) AS cost_per_conversion,
  CAST(NULL AS DOUBLE) AS roas,
  CAST(NULL AS DOUBLE) AS engagements,
  CAST(NULL AS DOUBLE) AS video_views,
  gold_processed_at
FROM {GOLD_SCHEMA}.rpt_meta_ad_performance_daily
UNION ALL
SELECT
  platform,
  full_date, year, month, month_name, day_name,
  account_id, account_name,
  campaign_id, campaign_name, campaign_status,
  campaign_channel_or_objective,
  campaign_daily_budget_inr,
  adset_or_adgroup_id,
  adset_or_adgroup_name,
  adset_or_adgroup_status,
  ad_id, ad_name,
  ad_type,
  ad_status,
  final_urls,
  impressions, clicks, ctr, spend_inr, cpc,
  conversions, conversions_value, cost_per_conversion, roas,
  engagements, video_views,
  gold_processed_at
FROM {GOLD_SCHEMA}.rpt_google_ad_performance_daily
""")
print("[OK] Gold.vw_unified_ad_performance")
print("[HINT] SELECT * FROM Gold.vw_unified_ad_performance WHERE platform = 'google_ads'")
print("[VALIDATE] google rpt", spark.table(f"{GOLD_SCHEMA}.rpt_google_ad_performance_daily").count())
print("[VALIDATE] unified", spark.sql(f"SELECT platform, COUNT(*) c FROM {GOLD_SCHEMA}.vw_unified_ad_performance GROUP BY platform").collect())
